# Day 053 — Exercise 1: Talk to the Backend

**What you'll build:** `check_health(client)` — the frontend's first HTTP call. It pings the backend's `GET /health` and returns `True` only if the backend answers `200` with `status == 'ok'`; any failure returns `False`.

**Why it matters:** Day 51 was a UI, Day 52 was an API. Today you connect them: the frontend talks to the backend over HTTP. The pattern that makes this testable is **an injected client** — your function takes a `client` object. In production it's an `httpx.Client`; in these checks it's a `TestClient` wrapping the backend in-process. Same code, no live server.

## Provided: Setup + the Backend (from Day 52)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import httpx
import ollama


# ---- The AI backend (built on Day 52 — provided here) ----
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    reply: str
    model: str


class HealthResponse(BaseModel):
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()


def build_api(model: str = 'llama3.2') -> FastAPI:
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app

## Your Implementation

In [ ]:
def check_health(client) -> bool:
    """
    GET /health via the injected client. Return True iff the response is
    200 AND its JSON status is 'ok'. Any exception -> False (never raises).
    """
    # TODO: try:
    #     resp = client.get('/health')
    #     return resp.status_code == 200 and resp.json().get('status') == 'ok'
    # TODO: except Exception:
    #     return False
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    backend = TestClient(build_api())

    # Check 1: healthy backend -> True
    try:
        assert check_health(backend) is True, 'expected True for a healthy backend'
        passed += 1; print('✅ Check 1: healthy backend -> True')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns an actual bool
    try:
        assert isinstance(check_health(backend), bool), 'must return a bool'
        passed += 1; print('✅ Check 2: returns a bool')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: connection failure -> False (no raise)
    try:
        class _Dead:
            def get(self, *a, **k):
                raise httpx.ConnectError('connection refused')
        assert check_health(_Dead()) is False, 'backend down must give False'
        passed += 1; print('✅ Check 3: backend down -> False (no crash)')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: wrong status value -> False
    try:
        degraded = FastAPI()
        @degraded.get('/health')
        def _h():
            return {'status': 'degraded'}
        assert check_health(TestClient(degraded)) is False, "status != 'ok' must give False"
        passed += 1; print('✅ Check 4: non-ok status -> False')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: missing /health route (404) -> False
    try:
        empty = TestClient(FastAPI())
        assert check_health(empty) is False, 'a 404 must give False'
        passed += 1; print('✅ Check 5: no /health route (404) -> False')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def check_health(client) -> bool:
    """Ping the backend's GET /health through an injected HTTP client.

    Returns True only if the request succeeds with 200 AND status == 'ok'.
    Any exception (backend down, connection refused) -> False, never raises.
    The `client` is duck-typed: an httpx.Client in production, a TestClient in
    tests — both expose .get / .post / .request.
    """
    try:
        resp = client.get('/health')
        return resp.status_code == 200 and resp.json().get('status') == 'ok'
    except Exception:
        return False
```

**Why this works:** The function never assumes the backend is up. It wraps the call in try/except so a refused connection becomes a clean `False` instead of a crash — exactly what a health badge needs. Taking `client` as a parameter (dependency injection) is the key move: the check cell passes a `TestClient(build_api())` that runs the backend in-process, while `frontend.py` will pass a real `httpx.Client(base_url=...)`. One function, both worlds.
</details>